# Context Engineering: Pruning Strategies Benchmark Notebook

This notebook demonstrates context engineering for long documents and compares four strategies:
- Raw Baseline (no pruning)
- Naive Regex Pruning
- Semantic Chunk Pruning
- LLMLingua Integration (with fallback if unavailable)

It captures: token usage, compression ratio, cost per query, latency, and LLM-as-a-Judge scores.

## How to use
1. Run all setup cells from top to bottom.
2. Upload a large text/PDF file and enter your question.
3. Click **Run Full Benchmark**.
4. Inspect comparison tables and dashboard charts.

## Concept Explainer

### 1) Raw Baseline (Gold Standard)
The full document is sent to the LLM. This is the reference answer quality, but usually the most expensive and slowest.

### 2) Naive Regex Pruning
A simple keyword and sentence filter reduces context by keeping only text likely relevant to the question. Fast but may drop critical facts.

### 3) Semantic Chunk Pruning
The document is chunked and ranked by semantic similarity to the question using embeddings. Better relevance than regex with controlled size.

### 4) LLMLingua Integration
Prompt compression is delegated to LLMLingua for structured token reduction while preserving answer-critical information.

### Evaluation Design
- **Efficiency**: token reduction, compression ratio, cost, latency
- **Quality**: judged against the baseline answer (Gold Standard) using LLM-as-a-Judge
- Metrics: contextual recall, contextual precision, answer correctness, faithfulness (1-10)

## Section A: Environment and Imports

This section prepares a Colab-first environment.

- Loads required data, pruning, and visualization packages
- Enables ipywidgets rendering in Colab
- Enables optional PDF parsing and LLMLingua compression

Run this once per Colab session before any downstream sections.

In [10]:
# Colab-first setup
# Installs lightweight dependencies needed for pruning, scoring, and rendering.
%pip install -q openai pandas numpy plotly ipywidgets scikit-learn pypdf matplotlib nbformat llmlingua

import importlib

# PDF parsing is optional; keep notebook runnable even if pypdf is unavailable.
try:
    importlib.import_module('pypdf')
    PDF_AVAILABLE = True
except Exception:
    PDF_AVAILABLE = False

# LLMLingua is optional; if unavailable, semantic fallback is used.
try:
    importlib.import_module('llmlingua')
    LLMLINGUA_AVAILABLE = True
except Exception:
    LLMLINGUA_AVAILABLE = False

# In Colab, this enables ipywidgets controls to render correctly.
try:
    colab_output = importlib.import_module('google.colab.output')
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete.")
print(f"pypdf: {PDF_AVAILABLE} | llmlingua(local): {LLMLINGUA_AVAILABLE}")

Note: you may need to restart the kernel to use updated packages.
Setup complete.
pypdf: True | llmlingua(local): True


## Section B: Core Pruning Engine and Utility Functions

This section introduces the core logic layer used in the next code cell.

It includes:
- Portable file extraction and cleaning
- Provider client creation and model eligibility checks
- Token and cost accounting
- LLM answer generation wrapper
- Naive, semantic, and LLMLingua pruning methods

Run the next code cell to register reusable helper functions.

### Core Section Notes

This is the reusable helper layer only.

It defines functions and does not execute the full benchmark by itself.

In [11]:
# -----------------------------
# Helpers and core methods
# -----------------------------
# Make this cell runnable even if setup cell was skipped.
import importlib
import os
import re
import time
import numpy as np
import ipywidgets as widgets
from typing import Dict, List, Tuple, Any, Optional
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

if 'PDF_AVAILABLE' not in globals():
    try:
        importlib.import_module('pypdf')
        PDF_AVAILABLE = True
    except Exception:
        PDF_AVAILABLE = False

# Prefer local PromptCompressor when available.
if 'LLMLINGUA_AVAILABLE' not in globals():
    try:
        importlib.import_module('llmlingua')
        LLMLINGUA_AVAILABLE = True
    except Exception:
        LLMLINGUA_AVAILABLE = False

# Lazy singleton to avoid re-initializing compressor each call.
PROMPT_COMPRESSOR = None

# Pricing table is intentionally zeroed so cost columns remain available without billing assumptions.
PRICING_USD_PER_1K = {
    ('xai', 'grok-4.1-fast-non-reasoning'): {'input': 0.0002, 'output': 0.0005},
    ('xai', 'grok-4.1-fast-reasoning'): {'input': 0.0002, 'output': 0.0005},
    ('xai', 'grok-4.3'): {'input': 0.00125, 'output': 0.0025}
}

# Stores a Colab-uploaded file when the fallback upload button is used.
COLAB_UPLOAD_ITEM = None


def get_uploaded_text(upload_widget: widgets.FileUpload, colab_upload_item: Optional[Dict[str, Any]] = None) -> Tuple[str, str]:
    # Prefer explicit Colab upload payload; otherwise read from ipywidgets uploader.
    item = colab_upload_item
    if item is None:
        value = upload_widget.value
        if isinstance(value, dict) and value:
            item = next(iter(value.values()))
        elif isinstance(value, (list, tuple)) and len(value) > 0:
            item = value[0]
        else:
            raise ValueError('Upload a file first.')

    name = item.get('name') or item.get('metadata', {}).get('name') or 'uploaded_document.txt'
    content = item.get('content')
    if content is None:
        raise ValueError('Uploaded file content is missing.')

    # Parse PDF bytes when needed; otherwise decode as UTF-8 text.
    if name.lower().endswith('.pdf'):
        if not PDF_AVAILABLE:
            raise ImportError('Install pypdf to parse PDF files.')
        from io import BytesIO
        from pypdf import PdfReader
        reader = PdfReader(BytesIO(content))
        text = '\n'.join(page.extract_text() or '' for page in reader.pages)
    else:
        text = content.decode('utf-8', errors='ignore')

    # Normalize whitespace to improve downstream chunking/scoring consistency.
    clean = re.sub(r'\s+', ' ', text).strip()
    return name, clean


def create_xai_client(inline_api_key: str = '') -> OpenAI:
    # UI key takes precedence; fallback to XAI_API_KEY environment variable.
    api_key = inline_api_key.strip() or os.getenv('XAI_API_KEY', '').strip()
    if not api_key:
        raise EnvironmentError('Add XAI_API_KEY in env or paste it in the UI field.')
    return OpenAI(
        api_key=api_key,
        base_url='https://api.x.ai/v1'
    )


def token_count(text: str) -> int:
    # Fast approximation for comparative benchmarking (not exact tokenizer count).
    if not text:
        return 0
    return int(len(text.split()) * 1.3)


def compute_cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    prices = PRICING_USD_PER_1K.get(('xai', model), {'input': 0.0, 'output': 0.0})
    return (input_tokens / 1000.0) * prices['input'] + (output_tokens / 1000.0) * prices['output']


def _extract_text_and_usage(chat_resp, prompt: str) -> Tuple[str, int, int]:
    # Read token usage from API response when available; fallback to estimated counts.
    answer = chat_resp.choices[0].message.content if chat_resp.choices else ''
    usage = getattr(chat_resp, 'usage', None)
    in_tok = getattr(usage, 'prompt_tokens', token_count(prompt))
    out_tok = getattr(usage, 'completion_tokens', token_count(answer))
    return answer, in_tok, out_tok


def call_llm_answer(client: OpenAI, model: str, context: str, question: str) -> Tuple[str, int, int, float]:
    prompt = f"You are a precise analyst. Answer strictly from the provided context.\n\nContext:\n{context}\n\nQuestion: {question}\n\nIf information is missing, say so clearly."

    # Track end-to-end model latency for each method.
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': 'You answer with concise, factual, context-grounded responses.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0.2
    )
    latency = time.perf_counter() - start

    answer, in_tok, out_tok = _extract_text_and_usage(resp, prompt)
    return answer, in_tok, out_tok, latency


def sentence_split(text: str) -> List[str]:
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


def naive_regex_prune(text: str, question: str, max_sentences: int = 120) -> str:
    # Keep sentences that match question keywords; fallback to leading sentences.
    sentences = sentence_split(text)
    keywords = [w.lower() for w in re.findall(r'[A-Za-z]{3,}', question)]
    if not keywords:
        return ' '.join(sentences[:max_sentences])

    keep = []
    for s in sentences:
        s_low = s.lower()
        if any(k in s_low for k in keywords):
            keep.append(s)

    if not keep:
        keep = sentences[:max_sentences]

    return ' '.join(keep[:max_sentences])


def chunk_text(text: str, chunk_size_words: int = 180, overlap_words: int = 40) -> List[str]:
    # Sliding-window chunking to preserve local context across chunk boundaries.
    words = text.split()
    if not words:
        return []

    chunks = []
    step = max(1, chunk_size_words - overlap_words)
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size_words]
        if chunk:
            chunks.append(' '.join(chunk))
    return chunks


def semantic_chunk_prune(text: str, question: str, top_k: int = 8) -> str:
    # Rank chunks by TF-IDF cosine similarity to the question and keep top-k.
    chunks = chunk_text(text)
    if not chunks:
        return ''

    vect = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    vect.fit([question] + chunks)

    q_vec = vect.transform([question])
    c_vec = vect.transform(chunks)
    scores = cosine_similarity(q_vec, c_vec).flatten()

    idx = np.argsort(scores)[::-1][:min(top_k, len(chunks))]
    selected = [chunks[i] for i in idx]
    return '\n\n'.join(selected)


def llmlingua_prune(text: str, question: str, target_ratio: float = 0.35) -> str:
    # Local LLMLingua path (no xAI API call). Falls back to semantic pruning if unavailable.
    global PROMPT_COMPRESSOR

    target_words = max(150, int(len(text.split()) * target_ratio))
    target_tokens = max(200, int(token_count(text) * target_ratio))

    if LLMLINGUA_AVAILABLE:
        try:
            if PROMPT_COMPRESSOR is None:
                from llmlingua import PromptCompressor
                PROMPT_COMPRESSOR = PromptCompressor(
                    model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
                    use_llmlingua2=True,
                    device_map="cpu"
                )

            compression_input = f"Context:\n{text}\n\nQuestion: {question}"
            compress_fn = getattr(PROMPT_COMPRESSOR, 'compress_prompt')
            compressed = compress_fn(
                context=compression_input,
                instruction='Keep only information required to answer the question accurately.',
                question=question,
                target_token=target_tokens
            )

            if isinstance(compressed, dict):
                compressed_text = (
                    compressed.get('compressed_prompt')
                    or compressed.get('prompt')
                    or compressed.get('text')
                    or ''
                )
            else:
                compressed_text = str(compressed)

            if compressed_text.strip():
                # Hard-cap response length to target budget for fair method comparison.
                return ' '.join(compressed_text.split()[:target_words])
        except Exception:
            # Fall back to semantic pruning when local compressor fails.
            pass

    reduced = semantic_chunk_prune(text, question, top_k=5)
    words = reduced.split()
    return ' '.join(words[:target_words])


def build_method_contexts(raw_text: str, question: str) -> Dict[str, str]:
    # Build all contexts once so each answering method receives a deterministic input.
    return {
        'Raw Baseline': raw_text,
        'Naive': naive_regex_prune(raw_text, question),
        'Semantic': semantic_chunk_prune(raw_text, question),
        'LLMLingua': llmlingua_prune(raw_text, question)
    }

## Section C: LLM-as-a-Judge and Visualization Builder

This section turns raw outputs into quality evaluation and visual storytelling.

- LLM judge scoring against the Gold Standard
- Structured JSON score parsing
- Multi-panel dashboard creation (kept as optional utility)

Run this once to register evaluator and charting functions.

In [12]:
# -----------------------------
# Judge and dashboard functions
# -----------------------------
import json
import re as regex
from typing import Dict
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from openai import OpenAI


def llm_judge_scores(client: OpenAI, judge_model: str, question: str, gold_answer: str, candidate_answer: str) -> Dict[str, float]:
    # Judge prompt compares each candidate answer against the baseline answer.
    judge_prompt = f"You are an impartial evaluator.\n\nQuestion: {question}\n\nGold Standard Answer:\n{gold_answer}\n\nCandidate Answer:\n{candidate_answer}\n\nScore candidate from 1-10 for each metric:\n- Contextual Recall\n- Contextual Precision\n- Answer Correctness\n- Faithfulness\n\nReturn only valid JSON object with keys: contextual_recall, contextual_precision, answer_correctness, faithfulness."

    resp = client.chat.completions.create(
        model=judge_model,
        messages=[
            {'role': 'system', 'content': 'You are a strict evaluator. Return only JSON.'},
            {'role': 'user', 'content': judge_prompt}
        ],
        temperature=0
    )

    # Extract first JSON object defensively, even if model returns extra text.
    text = (resp.choices[0].message.content or '').strip() if resp.choices else ''
    match = regex.search(r'\{.*?\}', text, flags=regex.DOTALL)
    if not match:
        raise ValueError(f'Judge output is not JSON: {text}')

    data = json.loads(match.group(0))
    return {
        'Contextual Recall': float(data['contextual_recall']),
        'Contextual Precision': float(data['contextual_precision']),
        'Answer Correctness': float(data['answer_correctness']),
        'Faithfulness': float(data['faithfulness'])
    }


def create_dashboard(metrics_df: pd.DataFrame, judge_df: pd.DataFrame) -> go.Figure:
    # Kept for optional reuse, even though dashboard rendering is currently disabled.
    methods = metrics_df['Method'].tolist()
    colors = {
        'bg': '#f6f8fb',
        'paper': '#ffffff',
        'ink': '#0f172a',
        'accent1': '#0ea5a4',
        'accent2': '#fb7185',
        'accent3': '#f59e0b',
        'accent4': '#14b8a6'
    }

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            'Token Footprint',
            'Cost and Latency',
            'Compression Savings',
            'Quality Radar'
        ),
        specs=[[{'type': 'bar'}, {'type': 'bar'}], [{'type': 'bar'}, {'type': 'polar'}]]
    )

    # Panel 1: original vs compressed token footprint.
    fig.add_trace(
        go.Bar(name='Original Tokens', x=methods, y=metrics_df['Original Tokens'], marker_color=colors['accent1']),
        row=1,
        col=1
    )
    fig.add_trace(
        go.Bar(name='Compressed Tokens', x=methods, y=metrics_df['Compressed Tokens'], marker_color=colors['accent2']),
        row=1,
        col=1
    )

    # Panel 2: estimated cost + measured latency.
    fig.add_trace(
        go.Bar(name='Cost USD', x=methods, y=metrics_df['Cost USD'], marker_color=colors['accent3']),
        row=1,
        col=2
    )
    fig.add_trace(
        go.Scatter(
            name='Latency (s)',
            x=methods,
            y=metrics_df['Latency (s)'],
            mode='lines+markers',
            marker_color=colors['ink'],
            line=dict(width=3)
        ),
        row=1,
        col=2
    )

    # Panel 3: compression ratio by method.
    fig.add_trace(
        go.Bar(name='Compression Ratio %', x=methods, y=metrics_df['Compression Ratio %'], marker_color=colors['accent4']),
        row=2,
        col=1
    )

    judge_avg = judge_df.groupby('Method', as_index=False).mean(numeric_only=True)
    radar_metrics = ['Contextual Recall', 'Contextual Precision', 'Answer Correctness', 'Faithfulness']
    radar_palette = ['#14b8a6', '#fb7185', '#f59e0b', '#2563eb']

    # Panel 4: quality radar chart from judge metrics.
    for i, (_, r) in enumerate(judge_avg.iterrows()):
        fig.add_trace(
            go.Scatterpolar(
                r=[r[m] for m in radar_metrics],
                theta=radar_metrics,
                fill='toself',
                name=r['Method'],
                line=dict(color=radar_palette[i % len(radar_palette)], width=2)
            ),
            row=2,
            col=2
        )

    fig.update_layout(
        title='Context Engineering Benchmark Dashboard',
        title_x=0.5,
        barmode='group',
        height=980,
        paper_bgcolor=colors['paper'],
        plot_bgcolor=colors['bg'],
        font=dict(family='Trebuchet MS, Verdana, sans-serif', color=colors['ink'], size=13),
        legend_title='Methods',
        margin=dict(l=40, r=40, t=90, b=40)
    )

    fig.update_annotations(font=dict(size=14, color=colors['ink']))
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(gridcolor='#e2e8f0')
    fig.update_polars(
        bgcolor=colors['bg'],
        radialaxis=dict(range=[0, 10], tickvals=[2, 4, 6, 8, 10], gridcolor='#cbd5e1')
    )
    return fig

## Section D: End-to-End Benchmark Orchestrator

This section defines the callback that powers the benchmark run.

Execution flow:
1. Read uploaded file + question
2. Build contexts for all pruning methods
3. Generate answers and efficiency metrics
4. Judge each method against the baseline answer
5. Render result tables and method answers

After running this section, go to the final section to launch the UI and click **Run Full Benchmark**.

In [13]:
# -----------------------------
# Benchmark runner (button callback)
# -----------------------------
import textwrap
from typing import Any
import pandas as pd
import IPython.display as ip_display

LAST_RUN = {}
METHOD_ORDER = ['Raw Baseline', 'Naive', 'Semantic', 'LLMLingua']

# Forward declarations keep notebook linting happy before the UI cell runs.
status_out: Any = None
table_out: Any = None
judge_out: Any = None
file_uploader: Any = None
question_input: Any = None
answer_model_input: Any = None
judge_model_input: Any = None
xai_api_key_input: Any = None


def run_benchmark(_):
    # Reset output panes for a clean run.
    status_out.clear_output()
    table_out.clear_output()
    judge_out.clear_output()

    with status_out:
        print('Starting benchmark...')

    try:
        # Read uploaded content and validate question input.
        upload_item = COLAB_UPLOAD_ITEM if COLAB_UPLOAD_ITEM is not None else None
        filename, raw_text = get_uploaded_text(file_uploader, colab_upload_item=upload_item)
        question = question_input.value.strip()
        if not question:
            raise ValueError('Please enter a question.')

        answer_model = answer_model_input.value
        judge_model = judge_model_input.value
        inline_key = xai_api_key_input.value.strip()

        # Separate clients allow independent tuning in future if needed.
        answer_client = create_xai_client(inline_key)
        judge_client = create_xai_client(inline_key)

        with status_out:
            print(f'File: {filename}')
            print(f'Question: {question}')
            print(f'Answer model: {answer_model}')
            print(f'Judge model: {judge_model}')
            print('Building method contexts...')

        contexts = build_method_contexts(raw_text, question)
        original_tokens = token_count(raw_text)

        records = []
        answers = {}

        # Run the same question across each pruning method.
        for method in METHOD_ORDER:
            ctx = contexts[method]
            with status_out:
                print(f'Running method: {method}')

            comp_tokens = token_count(ctx)
            answer, in_tok, out_tok, latency = call_llm_answer(
                answer_client,
                answer_model,
                ctx,
                question
            )
            cost = compute_cost_usd(answer_model, in_tok, out_tok)

            reduction_pct = max(0.0, (1 - (comp_tokens / max(1, original_tokens))) * 100.0)
            if method == 'Raw Baseline':
                # Baseline is intentionally uncompressed.
                reduction_pct = 0.0

            answers[method] = answer
            records.append({
                'Method': method,
                'Original Tokens': original_tokens,
                'Compressed Tokens': comp_tokens,
                'Compression Ratio %': round(reduction_pct, 2),
                'Input Tokens (Billed)': in_tok,
                'Output Tokens (Billed)': out_tok,
                'Cost USD': round(cost, 6),
                'Latency (s)': round(latency, 3)
            })

        metrics_df = pd.DataFrame(records)
        gold_answer = answers['Raw Baseline']

        with status_out:
            print('Running LLM-as-a-Judge scoring...')

        # Judge every method against the baseline answer.
        judge_rows = []
        for method in METHOD_ORDER:
            score = llm_judge_scores(
                judge_client,
                judge_model,
                question,
                gold_answer,
                answers[method]
            )
            judge_rows.append({'Method': method, **score})

        judge_df = pd.DataFrame(judge_rows)[[
            'Method',
            'Contextual Recall',
            'Contextual Precision',
            'Answer Correctness',
            'Faithfulness'
        ]]

        # Cache latest run data for quick inspection/export in later cells.
        LAST_RUN.clear()
        LAST_RUN.update({
            'metrics_df': metrics_df,
            'judge_df': judge_df,
            'answers': answers,
            'gold_answer': gold_answer,
            'question': question,
            'file': filename,
            'answer_model': answer_model,
            'judge_model': judge_model
        })

        with table_out:
            ip_display.display(ip_display.Markdown('### Comparison Table'))
            try:
                ip_display.display(metrics_df.style.background_gradient(cmap='YlGnBu', subset=['Compressed Tokens', 'Compression Ratio %', 'Latency (s)', 'Cost USD']))
            except Exception:
                ip_display.display(metrics_df)

        with judge_out:
            ip_display.display(ip_display.Markdown('### LLM-as-a-Judge Scores (1-10)'))
            ip_display.display(ip_display.Markdown(
                '- **Contextual Recall**: how much of the gold-standard information is preserved in the answer.\n'
                '- **Contextual Precision**: how much of the answer stays focused on relevant gold-standard information without extra noise.\n'
                '- **Answer Correctness**: how accurately the answer responds to the question compared with the gold standard.\n'
                '- **Faithfulness**: how well the answer stays grounded in the provided context without inventing facts.'
            ))
            try:
                ip_display.display(judge_df.style.background_gradient(cmap='YlOrBr', subset=['Contextual Recall', 'Contextual Precision', 'Answer Correctness', 'Faithfulness']))
            except Exception:
                ip_display.display(judge_df)

            ip_display.display(ip_display.Markdown('### Gold Standard (Raw Baseline) Answer'))
            print(textwrap.shorten(gold_answer, width=3000, placeholder=' ...[truncated]'))

            # Print only non-baseline answers because the gold standard is already shown above.
            ip_display.display(ip_display.Markdown('### Method Answers'))
            for method in METHOD_ORDER:
                if method == 'Raw Baseline':
                    continue
                ip_display.display(ip_display.Markdown(f'#### {method}'))
                print(textwrap.shorten(answers.get(method, ''), width=3000, placeholder=' ...[truncated]'))
                print('')

        with status_out:
            print('Benchmark complete.')

    except Exception as e:
        msg = str(e)
        with status_out:
            # Provide actionable guidance for the most common Gemini permission errors.
            if '403' in msg and 'Gemini API has not been used' in msg:
                print('Error: Gemini API is not enabled for this Google Cloud project.')
                print('Fix steps:')
                print('1) Open Google Cloud Console for the same project used by this API key.')
                print('2) Enable "Generative Language API" (Gemini API).')
                print('3) If prompted, enable billing for the project.')
                print('4) Wait 1-2 minutes, then rerun this cell and click Run Full Benchmark again.')
            elif '403' in msg and 'permission' in msg.lower():
                print('Error: API key lacks permission for Gemini API in this project.')
                print('Check API restrictions, project selection, and whether Generative Language API is enabled.')
            else:
                print(f'Error: {type(e).__name__}: {e}')


print('Runner ready. Launch the final UI section and click Run Full Benchmark.')

Runner ready. Launch the final UI section and click Run Full Benchmark.


## Section E: Launch UI (Final Step)

Upload and run controls are intentionally placed at the end so all helper functions are already loaded.

1. Upload a document
2. Enter your question
3. (Optional) paste xAI API key
4. Click **Run Full Benchmark**

In [17]:
# -----------------------------
# Final UI (Colab-first)
# -----------------------------
import importlib
import ipywidgets as widgets
from IPython.display import display

# Shared model list for both answer generation and judge scoring.
GROK_MODEL_OPTIONS = [
    'grok-4.1-fast-non-reasoning',
    'grok-4.1-fast-reasoning',
    'grok-4.3',
    'google/gemma-4-31b-it:free'
]

# Primary file input (works in most notebook environments).
file_uploader = widgets.FileUpload(
    accept='.txt,.md,.pdf',
    multiple=False,
    description='Upload File'
)

question_input = widgets.Textarea(
    placeholder='Ask a specific question about the uploaded document...',
    description='Question:',
    layout=widgets.Layout(width='100%', height='120px')
)

answer_model_input = widgets.Dropdown(
    options=GROK_MODEL_OPTIONS,
    value='grok-4.1-fast-non-reasoning',
    description='Answer Model:'
)

judge_model_input = widgets.Dropdown(
    options=GROK_MODEL_OPTIONS,
    value='grok-4.1-fast-non-reasoning',
    description='Judge Model:'
)

xai_api_key_input = widgets.Password(
    value='',
    description='xAI API Key:',
    placeholder='Optional: paste key here'
)

run_button = widgets.Button(
    description='Run Full Benchmark',
    button_style='success',
    icon='play'
)

# Secondary upload path for Colab sessions where FileUpload can be flaky.
colab_upload_button = widgets.Button(
    description='Colab Upload Fallback',
    button_style='info',
    icon='upload'
)


def colab_upload_file(_=None):
    global COLAB_UPLOAD_ITEM
    try:
        colab_files = importlib.import_module('google.colab.files')
    except Exception:
        print('Colab upload is only available in Google Colab.')
        return

    uploaded = colab_files.upload()
    if not uploaded:
        print('No file selected.')
        return

    name = next(iter(uploaded.keys()))
    COLAB_UPLOAD_ITEM = {
        'name': name,
        'metadata': {'name': name},
        'content': uploaded[name]
    }
    print(f'Colab upload loaded: {name}')


colab_upload_button.on_click(colab_upload_file)

# Output panes consumed by run_benchmark.
status_out = widgets.Output(layout={'border': '1px solid #cbd5e1'})
table_out = widgets.Output()
judge_out = widgets.Output()

# Remove previous callback when possible, then attach the latest runner.
try:
    run_button.on_click(run_benchmark, remove=True)
except TypeError:
    pass
run_button.on_click(run_benchmark)

# Compose a single vertical UI flow to keep interaction simple in Colab.
ui = widgets.VBox([
    widgets.HTML('<h2 style="margin:0;color:#0f172a;">Context Engineering Benchmark</h2>'),
    widgets.HTML('<p style="margin:0 0 8px 0;color:#334155;">Colab optimized workflow: upload file, ask question, run benchmark.</p>'),
    file_uploader,
    colab_upload_button,
    question_input,
    answer_model_input,
    judge_model_input,
    xai_api_key_input,
    run_button,
    widgets.HTML('<h4 style="color:#0f172a;">Status</h4>'),
    status_out,
    widgets.HTML('<h4 style="color:#0f172a;">Method Comparison</h4>'),
    table_out,
    widgets.HTML('<h4 style="color:#0f172a;">LLM-as-a-Judge Scores</h4>'),
    judge_out
])

display(ui)
print('UI ready. Upload file and run benchmark.')

UI ready. Upload file and run benchmark.


## Notes and Best Practices

- For stable pricing, keep a consistent model across all methods.
- LLMLingua availability depends on local package installation and model downloads.
- If using PDF, extraction quality can vary by document formatting.
- For very large files, consider adding a max context safety cap before baseline inference.

This notebook is designed as an experiment harness: swap in your own chunking policy, embedding model, or judge prompt as needed.